In [1]:
import pandas as pd

from MATRIX.models import BaseSurvival, CoxRegression, DeepSurv, RandomSurvForest, DeepMultiTask, CoxRegressionWithTimeVarying, DeepTimeVarying
from MATRIX.utils import get_data, get_metrics, get_results, get_xai, save_results

---

In [2]:
#REMOVE
#results = get_results(estimator_name="CoxRegression", dataset="colon.csv", seed=0)
#for result in results:
#    result.delete()

---

# Experiments

## Survival analysis (standard)

### Get data

In [2]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv")


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [3]:
modelCoxRegression = CoxRegression()
modelRandomSurvForest = RandomSurvForest(0)
modelDeepSurv = DeepSurv(X_train.shape[1])

### Fit and predict

In [4]:
modelCoxRegression = modelCoxRegression.fit(X_train, y_train)
modelRandomSurvForest = modelRandomSurvForest.fit(X_train, y_train)
modelDeepSurv = modelDeepSurv.fit(X_train, y_train)

survPredCoxRegression = modelCoxRegression.predict(X_test)
survPredRandomSurvForest = modelRandomSurvForest.predict(X_test)
survPredDeepSurv = modelDeepSurv.predict(X_test)

### Metrics

In [5]:
get_metrics([y_train, y_test], [survPredCoxRegression])

{'C-Index Harrel': 0.620846552012022,
 'C-Index IPCW': 0.6122561689357433,
 'Cumulative Dinamic AUC': [0.6285741381350445,
  0.6553820831596768,
  0.6558112662759072]}

In [6]:
get_metrics([y_train, y_test], [survPredRandomSurvForest])

{'C-Index Harrel': 0.6361245616964435,
 'C-Index IPCW': 0.6280965938985875,
 'Cumulative Dinamic AUC': [0.6317437061063123,
  0.6761228984690368,
  0.6878876204225357]}

In [7]:
get_metrics([y_train, y_test], [survPredDeepSurv])

{'C-Index Harrel': 0.5922107196526966,
 'C-Index IPCW': 0.5871237674458925,
 'Cumulative Dinamic AUC': [0.6092084229477996,
  0.6003116611430322,
  0.6445152151908579]}

### Survival / Cumulative Hazard Functions

In [ ]:
survival_function = modelCoxRegression.predict_survival_function(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=False)

In [ ]:
cumulative_hazard_function = modelCoxRegression.predict_cumulative_hazard_function(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [ ]:
modelCoxRegression.calculate_xai(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

## Survival analysis (multitask)

### Get data

In [11]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv", to_multitask=True)


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [12]:
modelDeepMultiTask = DeepMultiTask(X_train.shape[1])

### Fit and predict

In [13]:
modelDeepMultiTask = modelDeepMultiTask.fit(X_train, y_train)
survPredDeepMultiTask, binPredDeepMultiTask = modelDeepMultiTask.predict(X_test)

### Metrics

In [14]:
get_metrics([y_train, y_test], [survPredDeepMultiTask, binPredDeepMultiTask])

{'C-Index Harrel': 0.5150692937051261,
 'C-Index IPCW': 0.5077993684864526,
 'Cumulative Dinamic AUC': [0.5164806513956655,
  0.5102703562503049,
  0.5331083440800005],
 'MAE': 0.5191256830601093,
 'AMAE': 0.5191256830601093,
 'MS': 0.16939890710382513,
 'CCR': 0.4808743169398907,
 'RECALL0': 0.3114754098360656,
 'RECALL1': 0.16939890710382513}

### Survival / Cumulative Hazard Functions

In [15]:
survival_function = modelDeepMultiTask.predict_survival_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

In [16]:
cumulative_hazard_function = modelDeepMultiTask.predict_cumulative_hazard_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [17]:
modelDeepMultiTask.calculate_xai(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

## Survival analysis (time varying)

In [18]:
df = pd.read_csv("./MATRIX/datasets/colon.csv")

### Find splits dynamically

In [19]:
splits = BaseSurvival.dinamic_discretise(y=df[["time", "event"]], dataset="colon", seed=0, plot=False)

In [20]:
df["identifier"] = df.index.values

### Transform to time varying format

In [21]:
df = BaseSurvival.to_time_dependent(dataframe=df, splits=splits, identifier="identifier", time="time", event="event")

In [22]:
df = BaseSurvival.to_time_varying(dataframe=df, identifier="identifier", time="time", event="event")

### Get data

In [23]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(df=df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4049 entries, 0 to 4048
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   identifier    4049 non-null   int64  
 1   num_age       4049 non-null   int64  
 2   num_nodes     4049 non-null   float64
 3   fac_rx        4049 non-null   object 
 4   fac_sex       4049 non-null   int64  
 5   fac_differ    4049 non-null   object 
 6   fac_obstruct  4049 non-null   int64  
 7   fac_perfor    4049 non-null   int64  
 8   fac_adhere    4049 non-null   int64  
 9   fac_node4     4049 non-null   int64  
 10  event         4049 non-null   int64  
 11  time_start    4049 non-null   float64
 12  time_stop     4049 non-null   float64
dtypes: float64(3), int64(8), object(2)
memory usage: 411.4+ KB

         identifier      num_age    num_nodes   fac_rx      fac_sex  \
count   4049.000000  4049.000000  4049.000000     4049  4049.000000   
unique          NaN          NaN         

### Instantiate models

In [24]:
modelCoxRegressionWithTimeVarying = CoxRegressionWithTimeVarying()
modelDeepTimeVarying = DeepTimeVarying(X_train.shape[1])

### Fit and predict

In [25]:
modelCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.fit(X_train, y_train)
modelDeepTimeVarying = modelDeepTimeVarying.fit(X_train, y_train)

survPredCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.predict(X_test)
survPredDeepTimeVarying = modelDeepTimeVarying.predict(X_test)

### Metrics

In [26]:
get_metrics([y_train, y_test], [survPredCoxRegressionWithTimeVarying])

{'C-Index Harrel': 0.6706086623646506,
 'C-Index IPCW': 0.6585322447399417,
 'Cumulative Dinamic AUC': [0.6770340633108334,
  0.6695773476164759,
  0.6192424454026013]}

In [27]:
get_metrics([y_train, y_test], [survPredDeepTimeVarying])

{'C-Index Harrel': 0.5069452039811878,
 'C-Index IPCW': 0.5101314470110674,
 'Cumulative Dinamic AUC': [0.46800686588479573,
  0.5522360203619334,
  0.5214476025548159]}

### Survival / Cumulative Hazard Functions

In [28]:
survival_function = modelCoxRegressionWithTimeVarying.predict_survival_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

In [29]:
cumulative_hazard_function = modelCoxRegressionWithTimeVarying.predict_cumulative_hazard_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [30]:
modelCoxRegressionWithTimeVarying.calculate_xai(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

# Results

## Show stored results

In [2]:
results = get_results(estimator_name="CoxRegression", dataset="colon.csv")
#results = get_results(estimator_name="CoxRegressionWithTimeVarying", dataset="cgd.csv")
#results = get_results(estimator_name="DeepMultiTaskFFNN", dataset="colon.csv")

In [32]:
for i, result in enumerate(results):
    print(f"Result {i}:\n")
    print(f"    * Estimator: {result.config['estimator_name']} - Dataset: {result.config['dataset']} - Seed: {result.config['random_state']}")
    print(f"    * Best Params: {result.data_.best_params}")
    print()
    print(f"    * Metrics: {get_metrics(result.data_.targets, result.data_.predictions)}")
    print("\n\n")

Result 0:

    * Estimator: CoxRegression - Dataset: colon.csv - Seed: 0
    * Best Params: {'ties': 'efron', 'n_iter': 100, 'alpha': 10.0}

    * Metrics: {'C-Index Harrel': 0.6242694940724662, 'C-Index IPCW': 0.6263770389872937, 'Cumulative Dinamic AUC': [0.6319081880429113, 0.6625501526603059, 0.6657982681983687]}



Result 1:

    * Estimator: CoxRegression - Dataset: colon.csv - Seed: 1
    * Best Params: {'ties': 'efron', 'n_iter': 100, 'alpha': 10.0}

    * Metrics: {'C-Index Harrel': 0.6233711287865967, 'C-Index IPCW': 0.6325575070958646, 'Cumulative Dinamic AUC': [0.6635745207173778, 0.6375641306514759, 0.7276551159914055]}



Result 2:

    * Estimator: CoxRegression - Dataset: colon.csv - Seed: 2
    * Best Params: {'ties': 'efron', 'n_iter': 100, 'alpha': 10.0}

    * Metrics: {'C-Index Harrel': 0.6632091977005748, 'C-Index IPCW': 0.6380273777804222, 'Cumulative Dinamic AUC': [0.7159468476396074, 0.690291517955271, 0.7058971155871405]}



Result 3:

    * Estimator: CoxRegr

## Plot XAI results 

In [3]:
get_xai(estimator_name="CoxRegression", dataset="colon.csv", seed=0, individual=0)
#get_xai(estimator_name="CoxRegression", dataset="colon.csv")
#get_xai(dataset="colon.csv")

AttributeError: 'list' object has no attribute 'inverse_transform'

## Save stored results 

In [4]:
save_results(estimator_name="CoxRegression")